# 🔍 Live 3D Gaussian Splatting Densification Visualizer

Watch a 3DGS model **grow in real time**. A sparse set of Gaussians learns a synthetic scene while
[gsplat](https://github.com/nerfstudio-project/gsplat)'s `DefaultStrategy` clones, splits, and prunes them.

**Dashboard panels (updated every few steps):**

| Panel | What it shows |
|---|---|
| Render / Target | Current render vs. ground truth from an orbiting camera |
| 3D Gaussians | Gaussian centers. **Cyan = just cloned**, **orange = just split**, grey = older. Faint dots = target geometry |
| Count + PSNR | Number of Gaussians and image quality over time |
| Densification events | Per-refinement bars: clones (+), splits (+), prunes (−); dashed lines = opacity resets |
| Scale histogram | Max Gaussian scale by origin. Splitting shifts mass toward smaller Gaussians |

**Runtime → Change runtime type → GPU** before running.

## 1. Install

In [ ]:
# gsplat compiles its CUDA kernels on first use (JIT) - expect ~3-6 min the first time you render.
!pip install -q gsplat imageio imageio-ffmpeg

## 2. Imports & config

In [ ]:
import math, time, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import gridspec
from IPython.display import display, Video
import imageio.v2 as imageio

from gsplat import rasterization
from gsplat.strategy import DefaultStrategy
import gsplat.strategy.default as strat_mod

assert torch.cuda.is_available(), "Please switch to a GPU runtime (Runtime > Change runtime type)."
device = torch.device("cuda")

CFG = dict(
    # image / camera
    W=256, H=256, fov_deg=50, cam_radius=4.0, n_train_cams=48,
    # scenes
    n_teacher=6000,          # Gaussians in the hidden "ground-truth" scene
    n_init=300,              # Gaussians the student starts with
    # training
    steps=4000, viz_every=25, seed=0,
    # densification (gsplat DefaultStrategy) - play with these!
    refine_start=100, refine_stop=3000, refine_every=50, reset_every=1000,
    grow_grad2d=2e-4,        # lower -> more Gaussians get densified
    grow_scale3d=0.03,       # max-scale below this -> CLONE, above -> SPLIT
    prune_opa=0.005, prune_scale3d=0.25,
    # recording
    record=True, video_path="densification.mp4",
)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])

## 3. Synthetic target scene + cameras

In [ ]:
W, H = CFG["W"], CFG["H"]
R_CAM = CFG["cam_radius"]
focal = 0.5 * W / math.tan(math.radians(CFG["fov_deg"]) / 2)
K_single = torch.tensor([[focal, 0, W / 2], [0, focal, H / 2], [0, 0, 1]], dtype=torch.float32, device=device)

def lookat_viewmat(azim_deg, elev_deg, radius=R_CAM):
    """World->camera (OpenCV convention), camera on a sphere looking at the origin, z-up."""
    az, el = math.radians(azim_deg), math.radians(elev_deg)
    pos = radius * np.array([math.cos(el) * math.cos(az), math.cos(el) * math.sin(az), math.sin(el)])
    f = -pos / np.linalg.norm(pos)
    r = np.cross(f, [0, 0, 1.0]); r /= np.linalg.norm(r)
    d = np.cross(f, r)
    Rm = np.stack([r, d, f]); t = -Rm @ pos
    M = np.eye(4); M[:3, :3] = Rm; M[:3, 3] = t
    return torch.tensor(M, dtype=torch.float32, device=device)

def raster(means, quats, scales, opacities, colors, viewmats, Ks):
    img, alpha, info = rasterization(means, quats, scales, opacities, colors,
                                     viewmats, Ks, W, H, packed=False)
    return img, alpha, info

def make_teacher(n):
    """A rainbow trefoil-knot tube made of n small Gaussians."""
    t = torch.rand(n) * 2 * math.pi
    x = torch.sin(t) + 2 * torch.sin(2 * t)
    y = torch.cos(t) - 2 * torch.cos(2 * t)
    z = -torch.sin(3 * t)
    means = torch.stack([x, y, z], 1) * 0.4 + torch.randn(n, 3) * 0.05
    hue = t / (2 * math.pi)
    colors = 0.5 + 0.5 * torch.cos(2 * math.pi * (hue[:, None] + torch.tensor([0., 1 / 3, 2 / 3])))
    scales = 0.04 * (0.6 + 0.8 * torch.rand(n, 3))
    quats = F.normalize(torch.randn(n, 4), dim=-1)
    opac = torch.full((n,), 0.9)
    return {k: v.to(device) for k, v in dict(means=means, quats=quats, scales=scales,
                                              opacities=opac, colors=colors).items()}

teacher = make_teacher(CFG["n_teacher"])
def render_teacher(vm, K):
    t = teacher
    return raster(t["means"], t["quats"], t["scales"], t["opacities"], t["colors"], vm, K)[0]

# training cameras + ground-truth images
n_cams = CFG["n_train_cams"]
train_vm = torch.stack([lookat_viewmat(np.random.uniform(0, 360), np.random.uniform(-40, 60)) for _ in range(n_cams)])
train_K = K_single[None].repeat(n_cams, 1, 1)
with torch.no_grad():
    gt_train = render_teacher(train_vm, train_K)          # [N,H,W,3]
print("GT images:", tuple(gt_train.shape))

## 4. Student Gaussians, optimizers, densification strategy + event tracker

In [ ]:
def init_params(n):
    return nn.ParameterDict({
        "means":     nn.Parameter(((torch.rand(n, 3) * 2 - 1) * 1.5).to(device)),
        "scales":    nn.Parameter(torch.full((n, 3), math.log(0.2), device=device)),
        "quats":     nn.Parameter(F.normalize(torch.randn(n, 4), dim=-1).to(device)),
        "opacities": nn.Parameter(torch.full((n,), math.log(0.3 / 0.7), device=device)),   # logit
        "colors":    nn.Parameter((torch.randn(n, 3) * 0.1).to(device)),                    # pre-sigmoid
    })

def act(p):
    return p["means"], p["quats"], torch.exp(p["scales"]), torch.sigmoid(p["opacities"]), torch.sigmoid(p["colors"])

params = init_params(CFG["n_init"])
LR0_MEANS = 2e-3
lrs = dict(means=LR0_MEANS, scales=5e-3, quats=1e-3, opacities=5e-2, colors=2e-2)
optimizers = {k: torch.optim.Adam([{"params": params[k], "lr": lr, "name": k}], eps=1e-15) for k, lr in lrs.items()}

strategy = DefaultStrategy(
    refine_start_iter=CFG["refine_start"], refine_stop_iter=CFG["refine_stop"],
    refine_every=CFG["refine_every"], reset_every=CFG["reset_every"],
    grow_grad2d=CFG["grow_grad2d"], grow_scale3d=CFG["grow_scale3d"],
    prune_opa=CFG["prune_opa"], prune_scale3d=CFG["prune_scale3d"], verbose=False,
)
strategy.check_sanity(params, optimizers)
strat_state = strategy.initialize_state(scene_scale=1.0)

# ---------------------------------------------------------------------------------------------
# Event tracker: we wrap gsplat's low-level ops so we know *exactly* which Gaussians were
# cloned / split / pruned. These ops append/remove rows in a deterministic order:
#   duplicate: new rows appended at the end
#   split:     un-split rows kept, then 2 children per split parent appended
#   remove:    masked rows dropped
# ---------------------------------------------------------------------------------------------
class Tracker:
    ORIG, CLONE, SPLIT = 0, 1, 2
    def __init__(self, n):
        self.kind  = torch.zeros(n, dtype=torch.long, device=device)   # how this Gaussian was born
        self.birth = torch.zeros(n, dtype=torch.long, device=device)   # step at which it was born
        self.step = 0
        self.events, self.resets = [], []
        self.p_dup = self.p_split = self.p_prune = 0
        self.h_step, self.h_n, self.h_psnr = [], [], []
    def on_dup(self, mask):
        k = int(mask.sum())
        self.kind  = torch.cat([self.kind,  torch.full((k,), self.CLONE, device=device)])
        self.birth = torch.cat([self.birth, torch.full((k,), self.step,  device=device)])
        self.p_dup += k
    def on_split(self, mask):
        k, keep = int(mask.sum()), ~mask
        self.kind  = torch.cat([self.kind[keep],  torch.full((2 * k,), self.SPLIT, device=device)])
        self.birth = torch.cat([self.birth[keep], torch.full((2 * k,), self.step,  device=device)])
        self.p_split += k
    def on_remove(self, mask):
        keep = ~mask
        self.kind, self.birth = self.kind[keep], self.birth[keep]
        self.p_prune += int(mask.sum())
    def flush(self, step, n):
        if self.p_dup or self.p_split or self.p_prune:
            self.events.append((step, self.p_dup, self.p_split, self.p_prune))
            self.p_dup = self.p_split = self.p_prune = 0
        if len(self.kind) != n:   # safety net if a gsplat version does something unexpected
            print(f"[tracker] resync at step {step}: {len(self.kind)} -> {n}")
            self.kind  = torch.zeros(n, dtype=torch.long, device=device)
            self.birth = torch.full((n,), step, dtype=torch.long, device=device)

tr = Tracker(CFG["n_init"])

def _wrap(name, hook):
    orig = getattr(strat_mod, "_orig_" + name, None) or getattr(strat_mod, name)
    setattr(strat_mod, "_orig_" + name, orig)            # makes re-running this cell safe
    def f(*a, **kw):
        mask = kw["mask"] if "mask" in kw else (a[3] if len(a) > 3 else None)
        out = orig(*a, **kw)
        hook(mask)
        return out
    setattr(strat_mod, name, f)

_wrap("duplicate",  tr.on_dup)
_wrap("split",      tr.on_split)
_wrap("remove",     tr.on_remove)
_wrap("reset_opa",  lambda _: tr.resets.append(tr.step))
print("Student starts with", params["means"].shape[0], "Gaussians")

## 5. Dashboard

In [ ]:
plt.ioff()
plt.style.use("dark_background")
PALETTE = np.array([[0.42, 0.45, 0.50], [0.13, 0.83, 0.93], [0.98, 0.45, 0.09]])   # orig / clone / split
HIGHLIGHT_STEPS = 150                                      # how long "new" Gaussians stay highlighted
ghost = teacher["means"][::6].cpu().numpy()

fig = plt.figure(figsize=(16, 9), dpi=80)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.28, wspace=0.18, left=0.04, right=0.97, top=0.93, bottom=0.07)
ax_r, ax_g = fig.add_subplot(gs[0, 0]), fig.add_subplot(gs[0, 1])
ax3d = fig.add_subplot(gs[0, 2], projection="3d")
ax_n, ax_e, ax_s = fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1]), fig.add_subplot(gs[1, 2])
ax_p = ax_n.twinx()

@torch.no_grad()
def draw_dashboard(step):
    n = params["means"].shape[0]
    az, el = 30 + step * 0.15, 20
    vm, K = lookat_viewmat(az, el)[None], K_single[None]
    pred = raster(*act(params), vm, K)[0][0].clamp(0, 1)
    gt = render_teacher(vm, K)[0]
    psnr = float(-10 * torch.log10(F.mse_loss(pred, gt).clamp_min(1e-10)))
    tr.h_step.append(step); tr.h_n.append(n); tr.h_psnr.append(psnr)

    kind, birth = tr.kind.cpu().numpy(), tr.birth.cpu().numpy()
    means = params["means"].detach().cpu().numpy()
    age = step - birth
    recent = (kind > 0) & (age < HIGHLIGHT_STEPS)

    # 1/2: images
    for ax, im, title in [(ax_r, pred, f"Render  (PSNR {psnr:.1f} dB)"), (ax_g, gt, "Target")]:
        ax.clear(); ax.imshow(im.cpu().numpy()); ax.set_title(title); ax.axis("off")

    # 3: 3D centers
    ax3d.clear()
    ax3d.scatter(*ghost.T, s=1, c="#2a2a2a", depthshade=False, linewidths=0)
    idx_old = np.where(~recent)[0]
    idx_old = idx_old[::max(1, len(idx_old) // 5000)]
    idx_new = np.where(recent)[0][:4000]
    ax3d.scatter(*means[idx_old].T, s=3, c=PALETTE[0][None], alpha=0.55, depthshade=False, linewidths=0)
    if len(idx_new):
        ax3d.scatter(*means[idx_new].T, s=10, c=PALETTE[kind[idx_new]], depthshade=False, linewidths=0)
    ax3d.set_xlim(-2, 2); ax3d.set_ylim(-2, 2); ax3d.set_zlim(-2, 2)
    ax3d.view_init(elev=el, azim=az); ax3d.set_axis_off()
    ax3d.set_title(f"{n:,} Gaussians   ({int((recent & (kind==1)).sum())} new clones, "
                   f"{int((recent & (kind==2)).sum())} new splits)", fontsize=10)

    # 4: count + PSNR
    ax_n.clear(); ax_p.clear()
    ax_n.plot(tr.h_step, tr.h_n, color="#22d3ee", lw=2); ax_n.set_ylabel("# Gaussians", color="#22d3ee")
    ax_p.plot(tr.h_step, tr.h_psnr, color="#a3e635", lw=2); ax_p.set_ylabel("PSNR (dB)", color="#a3e635")
    ax_n.set_xlim(0, CFG["steps"]); ax_n.set_xlabel("step"); ax_n.set_title("Model size & quality")

    # 5: events
    ax_e.clear()
    if tr.events:
        s, d, sp, pr = (np.array(x) for x in zip(*tr.events))
        w = CFG["refine_every"] * 0.8
        ax_e.bar(s, d, w, color=PALETTE[1], label="clone")
        ax_e.bar(s, sp, w, bottom=d, color=PALETTE[2], label="split (parents)")
        ax_e.bar(s, -pr, w, color="#ef4444", label="prune")
        ax_e.legend(loc="upper right", fontsize=8)
    for r in tr.resets: ax_e.axvline(r, color="white", ls="--", lw=0.8, alpha=0.6)
    ax_e.set_xlim(0, CFG["steps"]); ax_e.set_xlabel("step"); ax_e.set_title("Densification events (dashed = opacity reset)")

    # 6: scale histogram by origin
    ax_s.clear()
    ls = torch.exp(params["scales"]).max(-1).values.log10().cpu().numpy()
    groups = [ls[kind == k] for k in (0, 1, 2)]
    ax_s.hist(groups, bins=40, range=(-3.0, 0.5), stacked=True, color=PALETTE,
              label=["initial", "cloned", "split"])
    ax_s.axvline(math.log10(CFG["grow_scale3d"]), color="white", ls=":", lw=1)
    ax_s.set_xlabel("log10(max scale)   (dotted: clone/split threshold)")
    ax_s.set_title("Gaussian size distribution"); ax_s.legend(fontsize=8)

    fig.suptitle(f"3DGS densification  -  step {step}/{CFG['steps']}", fontsize=14)
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()

## 6. Train with live visualization 🚀

In [ ]:
handle = display(fig, display_id=True)
writer = imageio.get_writer(CFG["video_path"], fps=12, macro_block_size=16) if CFG["record"] else None

t0 = time.time()
for step in range(CFG["steps"]):
    tr.step = step

    # ---- one training step on a random camera
    i = np.random.randint(CFG["n_train_cams"])
    img, alpha, info = raster(*act(params), train_vm[i:i + 1], train_K[i:i + 1])
    pred, gt = img[0], gt_train[i]
    loss = F.l1_loss(pred, gt) + 0.5 * F.mse_loss(pred, gt)

    strategy.step_pre_backward(params, optimizers, strat_state, step, info)
    loss.backward()

    for g in optimizers["means"].param_groups:                     # exponential lr decay on means
        g["lr"] = LR0_MEANS * (0.05 ** (step / CFG["steps"]))
    for opt in optimizers.values():
        opt.step(); opt.zero_grad(set_to_none=True)

    # ---- densification (clone / split / prune / opacity reset) happens in here
    strategy.step_post_backward(params, optimizers, strat_state, step, info, packed=False)
    tr.flush(step, params["means"].shape[0])

    # ---- live dashboard
    if step % CFG["viz_every"] == 0 or step == CFG["steps"] - 1:
        frame = draw_dashboard(step)
        handle.update(fig)
        if writer is not None: writer.append_data(frame)

if writer is not None: writer.close()
plt.close(fig)
print(f"Done in {time.time() - t0:.0f}s - final #Gaussians: {params['means'].shape[0]:,}, "
      f"PSNR: {tr.h_psnr[-1]:.1f} dB")

## 7. Replay / download the recording

In [ ]:
Video(CFG["video_path"], embed=True, width=900)

In [ ]:
# from google.colab import files; files.download(CFG["video_path"])

## Notes & ideas

* **Why the Gaussians behave the way they do** - `DefaultStrategy` accumulates the norm of the 2D-mean (screen-space) gradient per Gaussian. Every `refine_every` steps, Gaussians above `grow_grad2d` are **cloned** if small (`max scale ≤ grow_scale3d`) or **split** if large. Nearly transparent or huge Gaussians are **pruned**; `reset_every` periodically resets opacities.
* **Experiments:** lower `grow_grad2d` (more aggressive growth), raise `grow_scale3d` (more clones, fewer splits), set `reset_every` very large (no opacity resets), or swap in `gsplat.strategy.MCMCStrategy` for a fixed-budget approach.
* **Use your own scene:** replace section 3 with COLMAP-loaded cameras/images (see `examples/simple_trainer.py` in the gsplat repo) and initialize the student from the sparse point cloud. Everything else (tracker, dashboard, loop) stays the same - just feed it real `viewmats`, `Ks`, and GT images, and drop the "target ghost" points.
* **Slow dashboard?** Increase `viz_every`. Matplotlib drawing is the bottleneck, not training.